<a href="https://colab.research.google.com/github/sanjil18/CNN-project/blob/main/CNN_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import os
import shutil
import torch
from sklearn.model_selection import train_test_split
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

project_path = '/content/drive/MyDrive/CNN-project'
zip_path = f'{project_path}/intel-image-classification.zip'
data_dir = f'{project_path}/intel_data'
train_dir = f'{data_dir}/seg_train/seg_train'
val_dir = f'{data_dir}/seg_val'
test_dir = f'{data_dir}/seg_test/seg_test'

os.makedirs(project_path, exist_ok=True)

In [12]:
if os.path.exists(data_dir):
    print('Dataset ஏற்கனவே இருக்கு, download தேவையில்ல')
else:
    if not os.path.exists(zip_path):
        from google.colab import files
        files.upload()   # kaggle.json தேர்ந்தெடுங்க
        !mkdir -p ~/.kaggle
        !cp kaggle.json ~/.kaggle/
        !chmod 600 ~/.kaggle/kaggle.json
        !kaggle datasets download -d puneet6060/intel-image-classification -p "{project_path}"
    !unzip -q "{zip_path}" -d "{data_dir}"
    print('Dataset தயார்')

Dataset ஏற்கனவே இருக்கு, download தேவையில்ல


In [13]:
already_split = os.path.exists(val_dir) and len(os.listdir(val_dir)) > 0

if already_split:
    print('Split ஏற்கனவே முடிஞ்சிருக்கு, skip பண்றோம்')
else:
    os.makedirs(val_dir, exist_ok=True)
    for cls in os.listdir(train_dir):
        cls_path = os.path.join(train_dir, cls)
        images = os.listdir(cls_path)
        _, val_images = train_test_split(images, test_size=0.15, random_state=42)

        val_cls_dir = os.path.join(val_dir, cls)
        os.makedirs(val_cls_dir, exist_ok=True)

        for img_name in val_images:
            shutil.move(os.path.join(cls_path, img_name),
                        os.path.join(val_cls_dir, img_name))
        print(f'{cls}: moved {len(val_images)} images to val')

Split ஏற்கனவே முடிஞ்சிருக்கு, skip பண்றோம்


In [14]:
for name, path in [('train', train_dir), ('val', val_dir), ('test', test_dir)]:
    print(name)
    total = 0
    for cls in sorted(os.listdir(path)):
        n = len(os.listdir(os.path.join(path, cls)))
        total += n
        print('  ', cls, n)
    print('  total:', total)

train
   buildings 1862
   forest 1930
   glacier 2043
   mountain 2135
   sea 1932
   street 2024
  total: 11926
val
   buildings 329
   forest 341
   glacier 361
   mountain 377
   sea 342
   street 358
  total: 2108
test
   buildings 437
   forest 474
   glacier 553
   mountain 525
   sea 510
   street 501
  total: 3000


In [15]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [16]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_ds = datasets.ImageFolder(train_dir, transform=transform)
val_ds   = datasets.ImageFolder(val_dir, transform=transform)
test_ds  = datasets.ImageFolder(test_dir, transform=transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

print(train_ds.class_to_idx)
print(len(train_ds), len(val_ds), len(test_ds))

{'buildings': 0, 'forest': 1, 'glacier': 2, 'mountain': 3, 'sea': 4, 'street': 5}
11926 2108 3000


In [17]:
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 18 * 18, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN().to(device)
print(model)

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=41472, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=256, out_features=6, bias=True)
  )
)


In [18]:
images, labels = next(iter(train_loader))
images = images.to(device)
outputs = model(images)

print('Input shape :', images.shape)
print('Output shape:', outputs.shape)
print('Total params:', sum(p.numel() for p in model.parameters()))

Input shape : torch.Size([32, 3, 150, 150])
Output shape: torch.Size([32, 6])
Total params: 10711878


In [19]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

In [20]:
EPOCHS = 10
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
save_path = f'{project_path}/best_scratch_cnn.pth'

for epoch in range(EPOCHS):
    start = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(va_acc)

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), save_path)
        saved = '  ← best, saved'
    else:
        saved = ''

    print(f'Epoch {epoch+1}/{EPOCHS} | '
          f'train loss {tr_loss:.3f} acc {tr_acc:.3f} | '
          f'val loss {va_loss:.3f} acc {va_acc:.3f} | '
          f'{time.time()-start:.0f}s{saved}')

RuntimeError: stack expects each tensor to be equal size, but got [3, 150, 150] at entry 0 and [3, 76, 150] at entry 22

In [ ]:
transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_ds = datasets.ImageFolder(train_dir, transform=transform)
val_ds   = datasets.ImageFolder(val_dir, transform=transform)
test_ds  = datasets.ImageFolder(test_dir, transform=transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

print(len(train_ds), len(val_ds), len(test_ds))

In [21]:
print(train_ds.transform)

Compose(
    ToTensor()
    Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
)


In [22]:
from torchvision import transforms

new_transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_ds.transform = new_transform
val_ds.transform = new_transform
test_ds.transform = new_transform

print(train_ds.transform)

Compose(
    Resize(size=(150, 150), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
)


In [ ]:
bad = 0
for images, labels in train_loader:
    if images.shape[1:] != (3, 150, 150):
        bad += 1
print('Bad batches:', bad)